In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error

import pickle

In [2]:
import mlflow

mlflow.set_tracking_uri("sqlite:///C:/Users/mriyu/OneDrive/Desktop/DS2026/MLOPs_Taxi_Codesapce/mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment")

2026/03/26 21:33:26 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/03/26 21:33:26 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/03/26 21:33:26 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/03/26 21:33:26 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/03/26 21:33:26 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/03/26 21:33:26 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/03/26 21:33:26 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/03/26 21:33:26 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/03/26 21:33:26 INFO mlflow.tracking.fluent: Experiment with name 'nyc-taxi-experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:c:/Users/mriyu/OneDrive/Desktop/DS2026/MLOPs_Taxi_Codesapce/notebooks/mlruns/1', creation_time=1774541006483, experiment_id='1', last_update_time=1774541006483, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>

In [3]:
def read_dataframe(filename):
    if filename.endswith('.parquet'):
        df = pd.read_parquet(filename)

    elif filename.endswith('.csv'):
        df = pd.read_csv(filename)

        df['lpep_pickup_datetime'] = pd.to_datetime(df['lpep_pickup_datetime'])
        df['lpep_dropoff_datetime'] = pd.to_datetime(df['lpep_dropoff_datetime'])
        

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df['duration'] = df['duration'].dt.total_seconds() / 60
    df = df[(df['duration'] >= 1) & (df['duration'] <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df

In [5]:
df_train = read_dataframe('../data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('../data/green_tripdata_2021-02.parquet')

In [6]:
len(df_train), len(df_val)

(73908, 61921)

In [7]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

In [8]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [10]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [11]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

root_mean_squared_error(y_val, y_pred)

7.75871521021275

In [12]:
with open('../artifacts/lin_reg_model.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [40]:
# import mlflow

# mlflow.set_tracking_uri("file:./mlruns")
# mlflow.set_experiment("nyc-taxi-experiment")

# Day_05/25March2026 On "ominous-invention" codespace

In [14]:
import mlflow

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment")

with mlflow.start_run():
    mlflow.set_tag("developers","LearnerMP")

    mlflow.log_param("train-data-path", "../data/green_tripdata_2021-01.parquet")
    mlflow.log_param("val-data-path", "../data/green_tripdata_2021-02.parquet")

    alpha = 0.001
    mlflow.log_param("alpha", alpha)

    '''
    mlflow.log_param()

    This function logs a parameter (key-value pair) to MLflow
    Parameters are used to record:
    Input settings
    Configuration values
    File paths
    Hyperparameters

👉 These are not results, but inputs to your experiment
    '''

    lr = Lasso(alpha)
 

    lr.fit(X_train, y_train)
  

    y_pred_train = lr.predict(X_train)
    rmse_train = root_mean_squared_error(y_train, y_pred_train)
    mlflow.log_metric("rmse_train", rmse_train)

    y_pred_val = lr.predict(X_val)
    rmse_val = root_mean_squared_error(y_val, y_pred_val)
    mlflow.log_metric("rmse_val", rmse_val)

In [43]:
lr = Ridge()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

root_mean_squared_error(y_val, y_pred)

7.703735145138231

In [44]:
y_pred_on_train_data = lr.predict(X_train)
root_mean_squared_error(y_train, y_pred_on_train_data)

5.645810929583716